# Resolve GPS Coordinates from Google Maps Short Links

รันทีละ cell ตามลำดับ (กด ▶ หรือ Shift+Enter)

1. ติดตั้งไลบรารี
2. อัปโหลดไฟล์ `บันทึกการตรวจวัดปริมาณฝุ่น.xlsx`
3. ดึงพิกัด GPS จริงจาก short link ในคอลัมน์ B (ไล่ตาม redirect)
4. ดาวน์โหลดผลลัพธ์เป็น `gps_resolved.csv`

เสร็จแล้วส่ง `gps_resolved.csv` กลับมาให้ Claude เพื่อคำนวณ Moran's I / Getis-Ord Gi* / spatial covariates ใหม่ด้วยพิกัดจริง

In [1]:
# 1) ติดตั้งไลบรารีที่จำเป็น (Colab มี requests/openpyxl อยู่แล้วปกติ แต่รันบรรทัดนี้กันพลาด)
!pip install -q requests openpyxl

In [2]:
# 2) อัปโหลดไฟล์ .xlsx จากเครื่องของท่าน
from google.colab import files
uploaded = files.upload()  # เลือกไฟล์ "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx"
XLSX_PATH = list(uploaded.keys())[0]
print("อัปโหลดไฟล์แล้ว:", XLSX_PATH)

Saving บันทึกการตรวจวัดปริมาณฝุ่น.xlsx to บันทึกการตรวจวัดปริมาณฝุ่น.xlsx
อัปโหลดไฟล์แล้ว: บันทึกการตรวจวัดปริมาณฝุ่น.xlsx


In [3]:
# 3) ฟังก์ชันดึงพิกัดจาก short link (ไล่ตาม redirect ไปยัง URL ปลายทางจริง)
import re, time, csv
import openpyxl
import requests

SHEET_NAME = "Sheet1"
OUTPUT_CSV = "gps_resolved.csv"

# รูปแบบ URL ของ Google Maps ที่อาจเจอหลัง redirect และวิธีดึง lat/lon ออกมา
PATTERNS = [
    re.compile(r"@(-?\d+\.\d+),(-?\d+\.\d+)"),          # .../@14.1234,101.5678,17z
    re.compile(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)"),       # .../!3d14.1234!4d101.5678
    re.compile(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)"),      # ...?q=14.1234,101.5678
    re.compile(r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"),         # ...?ll=14.1234,101.5678
]

def extract_latlon(url_or_text: str):
    for pat in PATTERNS:
        m = pat.search(url_or_text)
        if m:
            return float(m.group(1)), float(m.group(2))
    return None, None

def resolve_link(short_url: str, timeout=10):
    try:
        resp = requests.get(short_url, allow_redirects=True, timeout=timeout,
                             headers={"User-Agent": "Mozilla/5.0"})
        final_url = resp.url
        lat, lon = extract_latlon(final_url)
        if lat is None:
            lat, lon = extract_latlon(resp.text[:5000])
        return lat, lon, final_url
    except Exception as e:
        return None, None, f"ERROR: {e}"

print("พร้อมแล้ว - ไปรัน cell ถัดไปเพื่อเริ่ม resolve")

พร้อมแล้ว - ไปรัน cell ถัดไปเพื่อเริ่ม resolve


In [4]:
# 4) อ่านไฟล์ .xlsx และ resolve ทีละจุด (32 จุด)
wb = openpyxl.load_workbook(XLSX_PATH, data_only=True)
ws = wb[SHEET_NAME]

rows_out = []
for row in ws.iter_rows(min_row=3, max_row=34, values_only=False):
    site_id = row[0].value
    link_cell = row[1].value
    addr = row[2].value
    if not site_id or not link_cell:
        continue
    print(f"Resolving {site_id} ...")
    lat, lon, final_url = resolve_link(link_cell)
    rows_out.append({
        "id": site_id,
        "address": addr,
        "short_link": link_cell,
        "resolved_url": final_url,
        "lat": lat,
        "lon": lon,
    })
    time.sleep(0.5)  # กันโดน rate-limit

n_ok = sum(1 for r in rows_out if r["lat"] is not None)
print(f"\nเสร็จแล้ว: resolve ได้ {n_ok}/{len(rows_out)} จุด")

Resolving จุดตรวจวัดอากาศ-1 ...
Resolving จุดตรวจวัดอากาศ-2 ...
Resolving จุดตรวจวัดอากาศ-3 ...
Resolving จุดตรวจวัดอากาศ-4 ...
Resolving จุดตรวจวัดอากาศ-5 ...
Resolving จุดตรวจวัดอากาศ-6 ...
Resolving จุดตรวจวัดอากาศ-7 ...
Resolving จุดตรวจวัดอากาศ-8 ...
Resolving จุดตรวจวัดอากาศ-9 ...
Resolving จุดตรวจวัดอากาศ-10 ...
Resolving จุดตรวจวัดอากาศ-11 ...
Resolving จุดตรวจวัดอากาศ-12 ...
Resolving จุดตรวจวัดอากาศ-13 ...
Resolving จุดตรวจวัดอากาศ-14 ...
Resolving จุดตรวจวัดอากาศ-15 ...
Resolving จุดตรวจวัดอากาศ-16 ...
Resolving จุดตรวจวัดอากาศ-17 ...
Resolving จุดตรวจวัดอากาศ-18 ...
Resolving จุดตรวจวัดอากาศ-19 ...
Resolving จุดตรวจวัดอากาศ-20 ...
Resolving จุดตรวจวัดอากาศ-21 ...
Resolving จุดตรวจวัดอากาศ-22 ...
Resolving จุดตรวจวัดอากาศ-23 ...
Resolving จุดตรวจวัดอากาศ-24 ...
Resolving จุดตรวจวัดอากาศ-25 ...
Resolving จุดตรวจวัดอากาศ-survey-0 ...
Resolving จุดตรวจวัดอากาศ-survey-1 ...
Resolving จุดตรวจวัดอากาศ-survey-2 ...
Resolving จุดตรวจวัดอากาศ-survey-3 ...
Resolving จุดตรวจวัดอากาศ-su

In [5]:
# 5) แสดงผลลัพธ์เป็นตาราง (เช็คก่อนดาวน์โหลด)
import pandas as pd
df = pd.DataFrame(rows_out)
df

,id,address,short_link,resolved_url,lat,lon
0,จุดตรวจวัดอากาศ-1,ตำบล ดอนดึง อำเภอ บ้านหมี่ ลพบุรี,https://goo.gl/maps/WfoDxiiEE7WW73dg7,https://www.google.com/maps/place/15%C2%B007'5...,15.132970,100.601207
1,จุดตรวจวัดอากาศ-2,ตำบล ดงมะรุม อำเภอ โคกสำโรง ลพบุรี,https://goo.gl/maps/ki5HpmiGYmLMLehp6,https://www.google.com/maps/place/15%C2%B007'5...,15.132317,100.857852
2,จุดตรวจวัดอากาศ-3,ตำบล ท่าดินดำ อำเภอชัยบาดาล ลพบุรี,https://goo.gl/maps/LKo17w2rPLGpyF1R9,https://www.google.com/maps/place/15%C2%B007'5...,15.131391,101.112316
3,จุดตรวจวัดอากาศ-4,ตำบล เขาน้อย อำเภอ ลำสนธิ ลพบุรี,https://goo.gl/maps/u8W3g37F66Gr1KYg7,https://www.google.com/maps/place/15%C2%B007'4...,15.130189,101.368495
4,จุดตรวจวัดอากาศ-5,ตำบล หินดาด อำเภอด่านขุนทด นครราชสีมา,https://goo.gl/maps/W1ouPwjpihGmuyqU8,https://www.google.com/maps/place/15%C2%B007'4...,15.128612,101.624558
5,จุดตรวจวัดอากาศ-6,ตำบล ท่าแค อำเภอเมืองลพบุรี ลพบุรี,https://goo.gl/maps/Z3n3LPXGJSb8ahe9A,https://www.google.com/maps/place/14%C2%B053'0...,14.884332,100.602652
6,จุดตรวจวัดอากาศ-7,ตำบล โคกตูม อำเภอเมืองลพบุรี ลพบุรี,https://goo.gl/maps/JvGuBtKRHAfgNWud7,https://www.google.com/maps/place/14%C2%B053'0...,14.884200,100.858153
7,จุดตรวจวัดอากาศ-8,ตำบล วังม่วง อำเภอ วังม่วง สระบุรี,https://goo.gl/maps/3CMy9Vas7P4rXAKX6,https://www.google.com/maps/place/14%C2%B053'0...,14.883863,101.112555
8,จุดตรวจวัดอากาศ-9,ตำบล ลำพญากลาง อำเภอมวกเหล็ก สระบุรี,https://goo.gl/maps/LE18dSbTMQAjN7RS6,https://www.google.com/maps/place/14%C2%B052'5...,14.882626,101.368374
9,จุดตรวจวัดอากาศ-10,ตำบล ลาดบัวขาว อำเภอสีคิ้ว นครราชสีมา,https://goo.gl/maps/7K2xm4WhbBm2KNqN6,https://www.google.com/maps/place/14%C2%B052'5...,14.882466,101.624613


In [6]:
# 6) บันทึกเป็น CSV แล้วดาวน์โหลดกลับเครื่องของท่าน
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
files.download(OUTPUT_CSV)
print("ดาวน์โหลด gps_resolved.csv แล้ว - ส่งไฟล์นี้กลับมาให้ Claude ได้เลย")

if n_ok < len(rows_out):
    print("\n\u26a0\ufe0f จุดที่ resolve ไม่ได้ (lat/lon ว่าง) ให้เปิด resolved_url ในคอลัมน์เพื่อดูพิกัดด้วยตาแล้วกรอกเพิ่มเอง")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ดาวน์โหลด gps_resolved.csv แล้ว - ส่งไฟล์นี้กลับมาให้ Claude ได้เลย
